# Manhattan Terrain Data

This notebook retrieves and processes public elevation data for Manhattan.

## Data Source

- Provider: City of New York and New York State GIS
- Dataset: 2017 NYC Topobathymetric LiDAR DEM
- Surface type: Bare-earth elevation
- Original resolution: 1 US survey foot
- Horizontal reference: New York Long Island State Plane
- Vertical reference: NAVD88
- Access: Public

In [1]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio

In [2]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

buildings_path = (
    project_root
    / "data"
    / "processed"
    / "manhattan_building_footprints_processed.gpkg"
)

buildings = gpd.read_file(
    buildings_path,
    layer="building_footprints"
)

print("Rows and columns:", buildings.shape)
print("Coordinate system:", buildings.crs)
print("File exists:", buildings_path.exists())

Rows and columns: (45195, 23)
Coordinate system: EPSG:2263
File exists: True


In [3]:
from urllib.parse import urlencode

## Retrieve the DEM Tile Index

The tile index identifies the locations and download links of the 2017 DEM files.

In [4]:
tile_index_base_url = (
    "https://elevation.its.ny.gov/arcgis/rest/services/"
    "Dem_Indexes/FeatureServer/0/query"
)

tile_index_params = {
    "where": "1=1",
    "outFields": (
        "FILENAME,SIZE_,DEM_COLLEC,"
        "YEAR,DIRECT_DL"
    ),
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "geojson"
}

tile_index_url = (
    f"{tile_index_base_url}?"
    f"{urlencode(tile_index_params)}"
)

dem_tiles = gpd.read_file(tile_index_url)

print("Rows and columns:", dem_tiles.shape)
print("Coordinate system:", dem_tiles.crs)
print("Columns:", dem_tiles.columns.tolist())

dem_tiles.head()

Rows and columns: (32, 6)
Coordinate system: EPSG:4326
Columns: ['FILENAME', 'SIZE_', 'DEM_COLLEC', 'YEAR', 'DIRECT_DL', 'geometry']


,FILENAME,SIZE_,DEM_COLLEC,YEAR,DIRECT_DL,geometry
0,be_NYC_001.tif,81613409.0,NYC,2017,https://gisdata.ny.gov/elevation/DEM/NYC_TopoB...,"POLYGON ((-74.21304 40.49881, -74.21307 40.509..."
1,be_NYC_002.tif,1558852.0,NYC,2017,https://gisdata.ny.gov/elevation/DEM/NYC_TopoB...,"POLYGON ((-74.21304 40.49881, -74.20502 40.505..."
2,be_NYC_004.tif,280444227.0,NYC,2017,https://gisdata.ny.gov/elevation/DEM/NYC_TopoB...,"POLYGON ((-74.21307 40.50957, -74.21326 40.568..."
3,be_NYC_005.tif,878255974.0,NYC,2017,https://gisdata.ny.gov/elevation/DEM/NYC_TopoB...,"POLYGON ((-74.1232 40.52563, -74.1233 40.57833..."
4,be_NYC_006.tif,174310192.0,NYC,2017,https://gisdata.ny.gov/elevation/DEM/NYC_TopoB...,"POLYGON ((-74.1232 40.52563, -74.09475 40.5467..."


In [5]:
from shapely.geometry import box

## Select DEM Tiles for Manhattan

Select only the DEM tiles that intersect the geographic extent of the Manhattan building footprints.

In [6]:
dem_tiles_projected = dem_tiles.to_crs(
    buildings.crs
)

building_bounds = buildings.total_bounds

manhattan_bounding_box = box(
    building_bounds[0],
    building_bounds[1],
    building_bounds[2],
    building_bounds[3]
)

manhattan_tiles = dem_tiles_projected.loc[
    dem_tiles_projected.intersects(
        manhattan_bounding_box
    )
].copy()

manhattan_tiles["size_mb"] = (
    manhattan_tiles["SIZE_"] / 1_000_000
)

print("Building bounds:")
print(building_bounds)

print(
    "\nNumber of intersecting tiles:",
    len(manhattan_tiles)
)

print(
    "Estimated total download size:",
    round(manhattan_tiles["size_mb"].sum(), 2),
    "MB"
)

manhattan_tiles[
    ["FILENAME", "size_mb", "DIRECT_DL"]
]

Building bounds:
[ 971135.9437198   188580.04971045 1010069.14941516  259423.2788914 ]

Number of intersecting tiles: 9
Estimated total download size: 5613.36 MB


,FILENAME,size_mb,DIRECT_DL
12,be_NYC_030.tif,962.539990,https://gisdata.ny.gov/elevation/DEM/NYC_TopoB...
15,be_NYC_033.tif,0.264162,https://gisdata.ny.gov/elevation/DEM/NYC_TopoB...
16,be_NYC_034.tif,819.089366,https://gisdata.ny.gov/elevation/DEM/NYC_TopoB...
22,be_NYC_019.tif,30.678870,https://gisdata.ny.gov/elevation/DEM/NYC_TopoB...
23,be_NYC_020.tif,819.589445,https://gisdata.ny.gov/elevation/DEM/NYC_TopoB...
24,be_NYC_021.tif,1037.106610,https://gisdata.ny.gov/elevation/DEM/NYC_TopoB...
27,be_NYC_025.tif,673.930486,https://gisdata.ny.gov/elevation/DEM/NYC_TopoB...
28,be_NYC_026.tif,998.951130,https://gisdata.ny.gov/elevation/DEM/NYC_TopoB...
31,be_NYC_029.tif,271.208716,https://gisdata.ny.gov/elevation/DEM/NYC_TopoB...


In [8]:
# Official NYC Borough Boundaries dataset
borough_endpoint = (
    "https://data.cityofnewyork.us/resource/"
    "gthc-hcne.geojson"
)

borough_params = {
    "$where": "borocode = 1"
}

borough_url = (
    f"{borough_endpoint}?{urlencode(borough_params)}"
)

# Download the Manhattan boundary
manhattan_boundary = gpd.read_file(borough_url)

# NOAA needs the upload boundary in latitude/longitude
# First project to feet, add a 500-foot buffer,
# then convert back to EPSG:4326.
download_aoi = manhattan_boundary.to_crs("EPSG:2263").copy()
download_aoi["geometry"] = download_aoi.geometry.buffer(500)
download_aoi = download_aoi.to_crs("EPSG:4326")

# Save only the name and geometry
aoi_path = (
    project_root
    / "data"
    / "raw"
    / "manhattan_dem_aoi.geojson"
)

download_aoi[
    ["boroname", "geometry"]
].to_file(
    aoi_path,
    driver="GeoJSON",
    index=False
)

print("Rows:", len(download_aoi))
print("Borough:", download_aoi["boroname"].tolist())
print("CRS:", download_aoi.crs)
print("Geometry:", download_aoi.geometry.geom_type.tolist())
print("File exists:", aoi_path.exists())
print("Saved to:", aoi_path)

Rows: 1
Borough: ['Manhattan']
CRS: EPSG:4326
Geometry: ['MultiPolygon']
File exists: True
Saved to: c:\Users\Owner\Desktop\曼岛flood risk\manhattan-pluvial-flood-risk\data\raw\manhattan_dem_aoi.geojson


In [9]:
import json

# Split Manhattan's MultiPolygon into individual Polygon features
aoi_parts = (
    download_aoi
    .explode(index_parts=False)
    .reset_index(drop=True)
)

# Build a minimal GeoJSON accepted by NOAA
geojson_data = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {},
            "geometry": geometry.__geo_interface__
        }
        for geometry in aoi_parts.geometry
    ]
}

# Overwrite the previous file
with aoi_path.open("w", encoding="utf-8") as file:
    json.dump(geojson_data, file)

print("Top-level fields:", list(geojson_data.keys()))
print("Number of polygons:", len(aoi_parts))
print("Geometry types:", aoi_parts.geometry.geom_type.unique())
print("File exists:", aoi_path.exists())

Top-level fields: ['type', 'features']
Number of polygons: 12
Geometry types: <StringArray>
['Polygon']
Length: 1, dtype: str
File exists: True


In [11]:
import json

geojson_data = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {},
            "geometry": single_aoi_4326.__geo_interface__
        }
    ]
}

with aoi_path.open("w", encoding="utf-8") as file:
    json.dump(geojson_data, file)

print("Top-level type:", geojson_data["type"])
print("Number of features:", len(geojson_data["features"]))
print(
    "Geometry type:",
    geojson_data["features"][0]["geometry"]["type"]
)
print("File exists:", aoi_path.exists())

Top-level type: FeatureCollection
Number of features: 1
Geometry type: Polygon
File exists: True


In [12]:
dem_vrt_url = (
    "https://noaa-nos-coastal-lidar-pds.s3.amazonaws.com/"
    "dem/NYC_topobathy_hydroEnforced_DEM_2017_9310/"
    "NYC_topobathy_hydroEnforced_DEM_2017_m9310_EPSG-6539.vrt"
)

# /vsicurl/ tells GDAL to access the raster through HTTP
dem_vrt_path = f"/vsicurl/{dem_vrt_url}"

with rasterio.Env(
    GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
    GDAL_HTTP_MULTIRANGE="YES"
):
    with rasterio.open(dem_vrt_path) as source:
        print("CRS:", source.crs)
        print("Pixel size:", source.res)
        print("Width and height:", source.width, source.height)
        print("Bounds:", source.bounds)
        print("NoData:", source.nodata)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 69: invalid continuation byte

Exception ignored in: 'rasterio._env.log_error'
Traceback (most recent call last):
  File "c:\Users\Owner\Desktop\曼岛flood risk\manhattan-pluvial-flood-risk\.venv\Lib\site-packages\rasterio\__init__.py", line 379, in open
    dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 69: invalid continuation byte


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 69: invalid continuation byte

In [13]:
import re
import tempfile
from pathlib import Path
from urllib.request import urlopen

# Folder containing the NOAA DEM tiles
dem_base_url = (
    "https://noaa-nos-coastal-lidar-pds.s3.amazonaws.com/"
    "dem/NYC_topobathy_hydroEnforced_DEM_2017_9310/"
)

dem_vrt_url = (
    dem_base_url
    + "NYC_topobathy_hydroEnforced_DEM_2017_m9310_EPSG-6539.vrt"
)

# Download only the small VRT index file
with urlopen(dem_vrt_url) as response:
    vrt_bytes = response.read()

try:
    vrt_text = vrt_bytes.decode("utf-8")
except UnicodeDecodeError:
    vrt_text = vrt_bytes.decode("latin-1")

if "<VRTDataset" not in vrt_text:
    raise ValueError("The downloaded file is not a valid GDAL VRT.")

# Find every source TIFF referenced by the VRT
source_pattern = re.compile(
    r"<SourceFilename[^>]*>([^<]+)</SourceFilename>"
)

source_files = source_pattern.findall(vrt_text)

# Replace relative filenames with complete NOAA /vsicurl/ paths
def replace_source(match):
    filename = match.group(1).strip().replace("\\", "/").split("/")[-1]

    return (
        '<SourceFilename relativeToVRT="0">'
        f"/vsicurl/{dem_base_url}{filename}"
        "</SourceFilename>"
    )

local_vrt_text = source_pattern.sub(
    replace_source,
    vrt_text
)

# Use the Windows temporary folder, whose path contains no Chinese characters
local_vrt_path = (
    Path(tempfile.gettempdir())
    / "nyc_hydroenforced_dem_2017.vrt"
)

local_vrt_path.write_text(
    local_vrt_text,
    encoding="utf-8"
)

print("VRT download size:", len(vrt_bytes), "bytes")
print("Referenced TIFF files:", len(source_files))
print("Temporary VRT:", local_vrt_path)
print("File exists:", local_vrt_path.exists())

VRT download size: 15136 bytes
Referenced TIFF files: 31
Temporary VRT: C:\Users\Owner\AppData\Local\Temp\nyc_hydroenforced_dem_2017.vrt
File exists: True


In [14]:
with rasterio.Env(
    GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
    GDAL_HTTP_MULTIRANGE="YES",
    CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif"
):
    with rasterio.open(local_vrt_path) as source:
        print("CRS:", source.crs)
        print("Pixel size:", source.res)
        print("Width and height:", source.width, source.height)
        print("Bounds:", source.bounds)
        print("NoData:", source.nodata)

CRS: EPSG:6539
Pixel size: (1.0000000000000002, 0.9999999999999999)
Width and height: 154627 154773
Bounds: BoundingBox(left=913082.0, bottom=118396.00000000748, right=1067709.0, top=273169.00000000745)
NoData: -3.4028230607370965e+38


In [15]:
from math import ceil
from rasterio.windows import Window, from_bounds

# One US survey foot equals this many metres
METERS_PER_US_SURVEY_FOOT = 1200 / 3937

target_resolution_m = 5
target_resolution_ft = (
    target_resolution_m
    / METERS_PER_US_SURVEY_FOOT
)

# Convert our AOI into the DEM coordinate system
aoi_6539 = gpd.GeoSeries(
    [single_aoi_4326],
    crs="EPSG:4326"
).to_crs("EPSG:6539")

aoi_bounds_6539 = aoi_6539.total_bounds

with rasterio.Env(
    GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
    GDAL_HTTP_MULTIRANGE="YES",
    CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif"
):
    with rasterio.open(local_vrt_path) as source:

        # Locate the AOI inside the complete 1-foot raster
        source_window = from_bounds(
            *aoi_bounds_6539,
            transform=source.transform
        )

        source_window = (
            source_window
            .round_offsets()
            .round_lengths()
        )

        # Prevent the window from extending beyond the dataset
        full_window = Window(
            0,
            0,
            source.width,
            source.height
        )

        source_window = source_window.intersection(
            full_window
        )

        # Calculate the dimensions after resampling to 5 metres
        output_width = ceil(
            source_window.width
            * source.res[0]
            / target_resolution_ft
        )

        output_height = ceil(
            source_window.height
            * source.res[1]
            / target_resolution_ft
        )

source_pixel_count = (
    source_window.width
    * source_window.height
)

output_pixel_count = (
    output_width
    * output_height
)

estimated_memory_mb = (
    output_pixel_count
    * 4
    / 1024**2
)

print(
    "Target resolution:",
    round(target_resolution_ft, 4),
    "US survey feet"
)
print(
    "Original AOI window:",
    int(source_window.width),
    "x",
    int(source_window.height),
    "pixels"
)
print(
    "Original pixels:",
    f"{source_pixel_count:,.0f}"
)
print(
    "5 m output:",
    output_width,
    "x",
    output_height,
    "pixels"
)
print(
    "Estimated uncompressed output:",
    round(estimated_memory_mb, 2),
    "MB"
)

Target resolution: 16.4042 US survey feet
Original AOI window: 40051 x 72465 pixels
Original pixels: 2,902,295,715
5 m output: 2442 x 4418 pixels
Estimated uncompressed output: 41.16 MB


In [16]:
from affine import Affine
from rasterio.enums import Resampling
from rasterio.features import geometry_mask

output_nodata = -9999.0

dem_output_path = (
    project_root
    / "data"
    / "processed"
    / "manhattan_dem_5m_epsg6539.tif"
)

print("Starting remote DEM read...")
print("Please keep the notebook running.")

with rasterio.Env(
    GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
    GDAL_HTTP_MULTIRANGE="YES",
    CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif",
    GDAL_HTTP_TIMEOUT="120",
    GDAL_HTTP_MAX_RETRY="3",
    GDAL_HTTP_RETRY_DELAY="2"
):
    with rasterio.open(local_vrt_path) as source:

        # Transform for the original 1-foot AOI window
        window_transform = source.window_transform(
            source_window
        )

        # Scale the transform to match the smaller 5 m array
        scale_x = source_window.width / output_width
        scale_y = source_window.height / output_height

        output_transform = (
            window_transform
            * Affine.scale(scale_x, scale_y)
        )

        # Read only the AOI window and aggregate it directly
        # to the target dimensions
        dem_5m = source.read(
            1,
            window=source_window,
            out_shape=(
                output_height,
                output_width
            ),
            resampling=Resampling.average,
            masked=True
        )

print("Remote read completed.")

# Mask pixels outside the buffered Manhattan AOI
inside_aoi = geometry_mask(
    [aoi_6539.iloc[0].__geo_interface__],
    out_shape=dem_5m.shape,
    transform=output_transform,
    invert=True
)

dem_5m = np.ma.masked_where(
    ~inside_aoi,
    dem_5m
)

# Also mask any invalid floating-point values
dem_5m = np.ma.masked_invalid(dem_5m)

# Convert to float32 and replace masked cells with -9999
dem_to_save = (
    dem_5m
    .filled(output_nodata)
    .astype("float32")
)

output_profile = {
    "driver": "GTiff",
    "height": output_height,
    "width": output_width,
    "count": 1,
    "dtype": "float32",
    "crs": "EPSG:6539",
    "transform": output_transform,
    "nodata": output_nodata,
    "compress": "deflate",
    "predictor": 3,
    "tiled": True,
    "blockxsize": 512,
    "blockysize": 512,
    "BIGTIFF": "IF_SAFER"
}

with rasterio.open(
    dem_output_path,
    "w",
    **output_profile
) as destination:

    destination.write(
        dem_to_save,
        1
    )

    destination.update_tags(
        source="NOAA 2017 NYC Topobathy HydroEnforced DEM",
        source_dataset_id="9310",
        horizontal_units="US survey feet",
        vertical_datum="NAVD88",
        vertical_units="US survey feet",
        nominal_resolution="5 metres",
        resampling_method="average"
    )

valid_elevations = dem_5m.compressed()

print("Saved to:", dem_output_path)
print("File exists:", dem_output_path.exists())
print(
    "File size:",
    round(
        dem_output_path.stat().st_size
        / 1024**2,
        2
    ),
    "MB"
)
print("Shape:", dem_5m.shape)
print(
    "Output pixel size:",
    output_transform.a,
    abs(output_transform.e),
    "US survey feet"
)
print(
    "Valid pixels:",
    f"{valid_elevations.size:,}"
)
print(
    "Elevation range:",
    round(float(valid_elevations.min()), 2),
    "to",
    round(float(valid_elevations.max()), 2),
    "feet NAVD88"
)

Starting remote DEM read...
Please keep the notebook running.


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 69: invalid continuation byte

Exception ignored in: 'rasterio._err.log_error'
Traceback (most recent call last):
  File "C:\Users\Owner\AppData\Local\Temp\ipykernel_19248\616394880.py", line 43, in <module>
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 69: invalid continuation byte


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 69: invalid continuation byte

Exception ignored in: 'rasterio._err.chaining_error_handler'
Traceback (most recent call last):
  File "C:\Users\Owner\AppData\Local\Temp\ipykernel_19248\616394880.py", line 43, in <module>
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 69: invalid continuation byte


...

Exception ignored in: 'rasterio._err.log_error'
Traceback (most recent call last):
  File "C:\Users\Owner\AppData\Local\Temp\ipykernel_19248\616394880.py", line 43, in <module>
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 69: invalid continuation byte


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 69: invalid continuation byte

Exception ignored in: 'rasterio._err.chaining_error_handler'
Traceback (most recent call last):
  File "C:\Users\Owner\AppData\Local\Temp\ipykernel_19248\616394880.py", line 43, in <module>
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 69: invalid continuation byte


Remote read completed.
Saved to: c:\Users\Owner\Desktop\曼岛flood risk\manhattan-pluvial-flood-risk\data\processed\manhattan_dem_5m_epsg6539.tif
File exists: True
File size: 0.24 MB
Shape: (4418, 2442)
Output pixel size: 16.400900900900904 16.402218198279762 US survey feet
Valid pixels: 0


ValueError: zero-size array to reduction operation minimum which has no identity

In [17]:
# NOAA's public S3 location
dem_s3_prefix = (
    "/vsis3/noaa-nos-coastal-lidar-pds/"
    "dem/NYC_topobathy_hydroEnforced_DEM_2017_9310/"
)

# Rewrite the VRT to use GDAL's S3 virtual filesystem
def replace_source_with_s3(match):
    filename = (
        match.group(1)
        .strip()
        .replace("\\", "/")
        .split("/")[-1]
    )

    return (
        '<SourceFilename relativeToVRT="0">'
        f"{dem_s3_prefix}{filename}"
        "</SourceFilename>"
    )

s3_vrt_text = source_pattern.sub(
    replace_source_with_s3,
    vrt_text
)

local_s3_vrt_path = (
    Path(tempfile.gettempdir())
    / "nyc_hydroenforced_dem_2017_s3.vrt"
)

local_s3_vrt_path.write_text(
    s3_vrt_text,
    encoding="utf-8"
)

# Select a small test block near the centre of the AOI
test_column = int(
    source_window.col_off
    + source_window.width / 2
)

test_row = int(
    source_window.row_off
    + source_window.height / 2
)

test_window = Window(
    test_column - 128,
    test_row - 128,
    256,
    256
)

print("Testing a small remote raster block...")

with rasterio.Env(
    AWS_NO_SIGN_REQUEST="YES",
    AWS_REGION="us-east-1",
    GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR"
):
    with rasterio.open(local_s3_vrt_path) as source:

        test_data = source.read(
            1,
            window=test_window,
            masked=True
        )

print("Test completed.")
print("Shape:", test_data.shape)
print("Valid pixels:", test_data.count())

if test_data.count() > 0:
    print(
        "Elevation range:",
        round(float(test_data.min()), 2),
        "to",
        round(float(test_data.max()), 2),
        "feet NAVD88"
    )

Testing a small remote raster block...


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 69: invalid continuation byte

Exception ignored in: 'rasterio._err.log_error'
Traceback (most recent call last):
  File "C:\Users\Owner\AppData\Local\Temp\ipykernel_19248\3379333811.py", line 64, in <module>
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 69: invalid continuation byte


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 69: invalid continuation byte

Exception ignored in: 'rasterio._err.chaining_error_handler'
Traceback (most recent call last):
  File "C:\Users\Owner\AppData\Local\Temp\ipykernel_19248\3379333811.py", line 64, in <module>
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 69: invalid continuation byte


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 69: invalid continuation byte

Exception ignored in: 'rasterio._err.log_error'
Traceback (most recent call last):
  File "C:\Users\Owner\AppData\Local\Temp\ipykernel_19248\3379333811.py", line 64, in <module>
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 69: invalid continuation byte


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 69: invalid continuation byte

Exception ignored in: 'rasterio._err.chaining_error_handler'
Traceback (most recent call last):
  File "C:\Users\Owner\AppData\Local\Temp\ipykernel_19248\3379333811.py", line 64, in <module>
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 69: invalid continuation byte


Test completed.
Shape: (256, 256)
Valid pixels: 0
